### Nazwa na kaggle:
`Karol Chądzyński`

### Załączone Submissions (2):
1. `2026-06-11___22:32:55___95.csv` 
    - Submission którego nie jestem w stanie z jakiegoś powodu oddtworzyć w pythonie, mimo zapisanych parametrów oraz tych samych features (gdzie jednak w ewentualnej ich zmianie domniemuje winy), jako że nie mam na nie "dowodów", załączam poniżej te na które mam dowód.
2. `2026-06-14___22:27:04___125.csv`
    - Submission które sprawuje się nieco gorzej, ale mam na nie dowód w pełni w pythonie.

### Dodatki:
- Plik był robiony w pythonie i tłumaczony z niego na R, mogą być między wynikami pewne różnice.
- Dlatego załączam poniżej link do repozytorium github, na którym znajduje się pełny kod pythona i wszystkie spushowane zmiany.
- Zgodnie z treścią projketu, nie załączam przeszukiwania przestrzeni hiperparametrów (były robione optuną w pythonie, znajdują się na repo), nie mniej submision (`2026-06-14___22:27:04___125.csv`) było z pozostawionych parametrów `best_params`.

### `best_params`

`best_params = {'max_depth': 6, 'n_estimators': 420, 'learning_rate': 0.03, 'subsample': 0.608, 'colsample_bytree': 0.546, 'colsample_bylevel': 0.76, 'colsample_bynode': 0.587, 'min_child_weight': 5, 'gamma': 0.001, 'max_delta_step': 1.3, 'reg_alpha': 0.14, 'reg_lambda': 3.878, 'tree_method': 'hist', 'random_state': 42}`

### Link do repozytorium:
- `https://github.com/Karolcha100/SAD_2026_proj2_in_python`

In [1]:
import pandas as pd
import numpy as np

RANDOM_STATE = 42

In [2]:
df_train = pd.read_csv(f"data-00-raw/bike_train.csv", index_col=[0], parse_dates=["date"])

df_train

,date,wcond,temp,atemp,hum,wind,count
1,2022-01-01,2,9.1,13.2,79.78,10.43,982
2,2022-01-02,2,9.9,12.7,68.91,16.16,797
3,2022-01-03,1,3.1,4.5,43.29,16.14,1351
4,2022-01-04,1,3.2,5.6,58.45,10.42,1559
5,2022-01-05,1,4.3,6.5,43.26,12.15,1597
...,...,...,...,...,...,...,...
361,2022-12-27,2,8.3,11.4,75.49,12.25,1161
362,2022-12-28,1,7.3,9.0,49.89,19.11,2301
363,2022-12-29,1,5.2,8.2,56.84,7.76,2421
364,2022-12-30,1,7.8,10.9,63.03,8.73,2995


In [3]:
df_test = pd.read_csv(f"data-00-raw/bike_test.csv", index_col=[0], parse_dates=["date"])

df_test

,date,wcond,temp,atemp,hum,wind
1,2023-01-01,1,10.2,13.8,68.56,12.49
2,2023-01-02,1,6.2,7.6,37.75,21.43
3,2023-01-03,1,1.1,1.3,43.68,23.77
4,2023-01-04,2,-0.6,1.0,41.04,12.01
5,2023-01-05,1,5.9,8.9,51.89,8.45
...,...,...,...,...,...,...
361,2023-12-27,3,5.0,6.0,81.51,20.58
362,2023-12-28,2,5.4,6.3,64.64,22.76
363,2023-12-29,2,5.4,7.8,58.41,10.11
364,2023-12-30,2,5.4,7.1,74.54,8.08


# Features Engineering

In [4]:
import holidays

In [5]:
def add_fourier_features(df: pd.DataFrame, n_harmonics: int = 3) -> pd.DataFrame:
    """Add Fourier seasonality terms for annual cycle.

    Parameters
    ----------
    df : pd.DataFrame
        DataFrame with ``dayofyear`` column.
    n_harmonics : int
        Number of harmonics to generate.

    Returns
    -------
    pd.DataFrame
        DataFrame with added sin/cos Fourier columns.
    """
    for k in range(1, n_harmonics + 1):
        df[f"sind{k}"] = np.sin(2 * np.pi * k * df["doy"] / 365)
        df[f"cosd{k}"] = np.cos(2 * np.pi * k * df["doy"] / 365)
    return df

def build_features(
        df: pd.DataFrame,
    ) -> pd.DataFrame:
    """Build full feature matrix from raw bike sharing data.

    Parameters
    ----------
    df : pd.DataFrame
        Raw DataFrame with columns: date, temp, atemp, hum, wind, wcond.

    Returns
    -------
    pd.DataFrame
        DataFrame with engineered features, without target column.
    """
    df = df.copy()
    df["date"] = pd.to_datetime(df["date"])

    df["month"] = df["date"].dt.month
    df["dow"] = df["date"].dt.dayofweek
    df["doy"] = df["date"].dt.dayofyear
    df["quart"] = df["date"].dt.quarter
    df["week"] = (df["dow"] >= 5).astype(int)
    df["doy_norm"] = df["doy"] / 365

    df["t"] = (df["date"] - pd.Timestamp("2022-01-01")).dt.days

    df["temp2"] = df["temp"] ** 2
    df["temp3"] = df["temp"] ** 3
    df["hum2"] = df["hum"] ** 2
    df["wind2"] = df["wind"] ** 2
    df["temp-atemp"] = df["temp"] - df["atemp"]

    df["tempxhum"] = df["temp"] * df["hum"]
    df["tempxwind"] = df["temp"] * df["wind"]
    df["atempxhum"] = df["atemp"] * df["hum"]
    df["humxwind"] = df["hum"] * df["wind"]
    df["wcondxtemp"] = df["wcond"] * df["temp"]

    # New Features
    ##############
    # df["is_summer"] = df["month"].isin([6, 7, 8]).astype(int)
    # df["is_winter"] = df["month"].isin([12, 1, 2]).astype(int)


    df["break_schools"] = df["month"].isin([6, 7, 8]).astype(int)
    df["break_academia"] = df["month"].isin([7, 8, 9]).astype(int)


    checker_holidays = holidays.Poland(years=df["date"].dt.year)
    df["holiday"] = df["date"].isin(checker_holidays).astype(int)
    
    # END New Features
    ##################

    wcond_dummies = pd.get_dummies(df["wcond"], prefix="wcond", drop_first=True)
    df = pd.concat([df, wcond_dummies], axis=1)

    df = add_fourier_features(df, n_harmonics=4)

    drop_cols = ["date", "wcond", "doy"]
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])

    return df

In [6]:
df_train_featured = df_train.copy()


df_train_featured = build_features(
    df_train_featured,
)

df_train_featured

,temp,atemp,hum,wind,count,month,dow,quart,week,doy_norm,...,wcond_2,wcond_3,sind1,cosd1,sind2,cosd2,sind3,cosd3,sind4,cosd4
1,9.1,13.2,79.78,10.43,982,1,5,1,1,0.002740,...,True,False,1.721336e-02,0.999852,3.442161e-02,0.999407,5.161967e-02,0.998667,6.880243e-02,0.997630
2,9.9,12.7,68.91,16.16,797,1,6,1,1,0.005479,...,True,False,3.442161e-02,0.999407,6.880243e-02,0.997630,1.031017e-01,0.994671,1.372788e-01,0.990532
3,3.1,4.5,43.29,16.14,1351,1,0,1,0,0.008219,...,False,False,5.161967e-02,0.998667,1.031017e-01,0.994671,1.543088e-01,0.988023,2.051045e-01,0.978740
4,3.2,5.6,58.45,10.42,1559,1,1,1,0,0.010959,...,False,False,6.880243e-02,0.997630,1.372788e-01,0.990532,2.051045e-01,0.978740,2.719582e-01,0.962309
5,4.3,6.5,43.26,12.15,1597,1,2,1,0,0.013699,...,False,False,8.596480e-02,0.996298,1.712931e-01,0.985220,2.553533e-01,0.966848,3.375229e-01,0.941317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,8.3,11.4,75.49,12.25,1161,12,1,4,0,0.989041,...,True,False,-6.880243e-02,0.997630,-1.372788e-01,0.990532,-2.051045e-01,0.978740,-2.719582e-01,0.962309
362,7.3,9.0,49.89,19.11,2301,12,2,4,0,0.991781,...,False,False,-5.161967e-02,0.998667,-1.031017e-01,0.994671,-1.543088e-01,0.988023,-2.051045e-01,0.978740
363,5.2,8.2,56.84,7.76,2421,12,3,4,0,0.994521,...,False,False,-3.442161e-02,0.999407,-6.880243e-02,0.997630,-1.031017e-01,0.994671,-1.372788e-01,0.990532
364,7.8,10.9,63.03,8.73,2995,12,4,4,0,0.997260,...,False,False,-1.721336e-02,0.999852,-3.442161e-02,0.999407,-5.161967e-02,0.998667,-6.880243e-02,0.997630


In [7]:
df_test_featured = df_test.copy()

df_test_featured = build_features(
    df_test_featured, 
)

df_test_featured

,temp,atemp,hum,wind,month,dow,quart,week,doy_norm,t,...,wcond_2,wcond_3,sind1,cosd1,sind2,cosd2,sind3,cosd3,sind4,cosd4
1,10.2,13.8,68.56,12.49,1,6,1,1,0.002740,365,...,False,False,1.721336e-02,0.999852,3.442161e-02,0.999407,5.161967e-02,0.998667,6.880243e-02,0.997630
2,6.2,7.6,37.75,21.43,1,0,1,0,0.005479,366,...,False,False,3.442161e-02,0.999407,6.880243e-02,0.997630,1.031017e-01,0.994671,1.372788e-01,0.990532
3,1.1,1.3,43.68,23.77,1,1,1,0,0.008219,367,...,False,False,5.161967e-02,0.998667,1.031017e-01,0.994671,1.543088e-01,0.988023,2.051045e-01,0.978740
4,-0.6,1.0,41.04,12.01,1,2,1,0,0.010959,368,...,True,False,6.880243e-02,0.997630,1.372788e-01,0.990532,2.051045e-01,0.978740,2.719582e-01,0.962309
5,5.9,8.9,51.89,8.45,1,3,1,0,0.013699,369,...,False,False,8.596480e-02,0.996298,1.712931e-01,0.985220,2.553533e-01,0.966848,3.375229e-01,0.941317
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,5.0,6.0,81.51,20.58,12,2,4,0,0.989041,725,...,False,True,-6.880243e-02,0.997630,-1.372788e-01,0.990532,-2.051045e-01,0.978740,-2.719582e-01,0.962309
362,5.4,6.3,64.64,22.76,12,3,4,0,0.991781,726,...,True,False,-5.161967e-02,0.998667,-1.031017e-01,0.994671,-1.543088e-01,0.988023,-2.051045e-01,0.978740
363,5.4,7.8,58.41,10.11,12,4,4,0,0.994521,727,...,True,False,-3.442161e-02,0.999407,-6.880243e-02,0.997630,-1.031017e-01,0.994671,-1.372788e-01,0.990532
364,5.4,7.1,74.54,8.08,12,5,4,1,0.997260,728,...,True,False,-1.721336e-02,0.999852,-3.442161e-02,0.999407,-5.161967e-02,0.998667,-6.880243e-02,0.997630


### X_[train|test] + y_[train] 

In [8]:
X = df_train_featured.drop(columns=["count"]).to_numpy()

y_log = np.log1p(df_train_featured["count"].to_numpy())

In [9]:
X_test = df_test_featured.to_numpy()

# Xgboost + Optuna

In [10]:
import xgboost as xgb
import optuna

from sklearn.metrics import root_mean_squared_error as rmse
from sklearn.model_selection import GroupKFold

In [11]:
kf = GroupKFold(n_splits=6)

groups: np.ndarray = df_train["date"].dt.isocalendar().week.astype(int).to_numpy()

In [12]:
def calculate_cv(X: np.ndarray, y_log: np.ndarray, params: dict[str, float]) -> float:
    val_rmses = []
    for train_idx, val_idx in kf.split(X, y_log, groups=groups):
        model = xgb.XGBRegressor(
            **params
        )

        model.fit(X[train_idx], y_log[train_idx])
        val_pred = np.expm1(model.predict(X[val_idx]))
        y_val_orig = np.expm1(y_log[val_idx])
        val_rmses.append(rmse(y_val_orig, val_pred))

    return np.average(val_rmses)

In [13]:
def average_cvrmse_by_seeds(
        best_params: dict[str, float], 
        seeds: list[int], 
        X: np.ndarray, y_log: np.ndarray,
        verbose: bool = False,
    ) -> tuple[list[np.ndarray], float]:
    best_params_noseed = {key: val for key, val in best_params.items() if key != "random_state"}

    RMSE_cv = []

    for seed in seeds:
        print(f"seed = {seed}", end = "\t") if verbose else None

        params = {**best_params_noseed, "random_state": seed}

        rmse = calculate_cv(X, y_log, params)
        RMSE_cv.append(rmse)

        print(f"[DONE]\t{rmse:.5f}", end = "\n")  if verbose else None

    
    avg_cv_rmse = np.average(RMSE_cv)
    print(f"seed = {"END"}\t[DONE]\tMean CV = {avg_cv_rmse:.5f}")  if verbose else None

    return float(avg_cv_rmse)

In [80]:
def objective(trial: optuna.Trial) -> float:
    params = {
        "max_depth": trial.suggest_int("max_depth", 2, 10), # zmienic na 6 jeżeli pzetrenowuje
        "n_estimators": trial.suggest_int("n_estimators", 50, 800), # zmieić na 400 jeśi przetrenowuje
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.5, log=True),

        "subsample": trial.suggest_float("subsample", 0.4, 0.8), 
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.8),
        "colsample_bylevel": trial.suggest_float("colsample_bylevel", 0.4, 0.8),
        "colsample_bynode": trial.suggest_float("colsample_bynode", 0.4, 0.8),

        "min_child_weight": trial.suggest_int("min_child_weight", 5, 30), 
        "gamma": trial.suggest_float("gamma", 0.0, 5.0), 
        "max_delta_step": trial.suggest_float("max_delta_step", 0.0, 5.0), 

        "reg_alpha": trial.suggest_float("reg_alpha", 0.1, 50.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 50.0, log=True),
        
        "tree_method": "hist",
        "random_state": RANDOM_STATE,
    }
    return average_cvrmse_by_seeds(
        params, 
        [seed_val for seed_val in range(5)],
        X, y_log, False
    )

In [81]:
study = optuna.create_study(
    study_name="xgb_bike",
    storage="sqlite:///params-optuna/optuna_xgb_new_features_avg_A1.db", 
    direction="minimize",
    load_if_exists=True, 
)

[I 2026-06-14 21:39:49,530] Using an existing study with name 'xgb_bike' instead of creating a new one.


In [ ]:
# study.optimize(objective, n_trials=300)

[I 2026-06-14 21:39:54,694] Trial 2489 finished with value: 582.1892665502908 and parameters: {'max_depth': 9, 'n_estimators': 634, 'learning_rate': 0.020530615967329777, 'subsample': 0.7414967966025987, 'colsample_bytree': 0.5600341293570729, 'colsample_bylevel': 0.6897049562240911, 'colsample_bynode': 0.6891307073248717, 'min_child_weight': 12, 'gamma': 0.084415671711526, 'max_delta_step': 3.9792151237446864, 'reg_alpha': 0.12357153369634645, 'reg_lambda': 9.482979544543221}. Best is trial 2282 with value: 516.2504570996613.
[I 2026-06-14 21:39:56,696] Trial 2490 finished with value: 546.9355522554613 and parameters: {'max_depth': 9, 'n_estimators': 630, 'learning_rate': 0.01461116054232925, 'subsample': 0.7462830937825596, 'colsample_bytree': 0.5407807640240083, 'colsample_bylevel': 0.6959359723153843, 'colsample_bynode': 0.6886310503904735, 'min_child_weight': 5, 'gamma': 0.09626970881134214, 'max_delta_step': 3.992439684530941, 'reg_alpha': 0.10001228621985232, 'reg_lambda': 9.287

In [142]:
print(f"[XGBoost + optuna]\nruns_params={study.best_params}\nbest_value={study.best_value}")

[XGBoost + optuna]
runs_params={'max_depth': 10, 'n_estimators': 673, 'learning_rate': 0.016275472726490006, 'subsample': 0.733562186336755, 'colsample_bytree': 0.5509434659946164, 'colsample_bylevel': 0.7037245542968691, 'colsample_bynode': 0.7010598868099925, 'min_child_weight': 5, 'gamma': 0.001268101935957147, 'max_delta_step': 3.779634460282497, 'reg_alpha': 0.10013041967270879, 'reg_lambda': 10.479085696096853}
best_value=514.7725556781845


In [159]:
best_params = study.best_params

best_params=dict(
    zip(
        ['max_depth', 'n_estimators', 'learning_rate', 'subsample', 'colsample_bytree', 'colsample_bylevel', 'colsample_bynode', 'min_child_weight', 'gamma', 'max_delta_step', 'reg_alpha', 'reg_lambda', 'tree_method', 'random_state'],
        [6,420,0.0300,0.6080,0.5460,0.7600,0.5870,5,0.0010,1.3000,0.1400,3.8780,'hist',42,]
,
    )
)

best_params = {**best_params, "tree_method": "hist", "random_state": RANDOM_STATE,}

In [160]:
print(f"best_params = {best_params}")

best_params = {'max_depth': 6, 'n_estimators': 420, 'learning_rate': 0.03, 'subsample': 0.608, 'colsample_bytree': 0.546, 'colsample_bylevel': 0.76, 'colsample_bynode': 0.587, 'min_child_weight': 5, 'gamma': 0.001, 'max_delta_step': 1.3, 'reg_alpha': 0.14, 'reg_lambda': 3.878, 'tree_method': 'hist', 'random_state': 42}


### Final Validation + Prediction

In [137]:
def average_multiple_models(
        best_params: dict[str, float], 
        seeds: list[int], 
        X: np.ndarray, y_log: np.ndarray, X_test: np.ndarray,
        verbose: bool = False,
    ) -> tuple[list[np.ndarray], float]:
    best_params_noseed = {key: val for key, val in best_params.items() if key != "random_state"}

    preds = []
    RMSE_cv = []

    for seed in seeds:
        print(f"seed = {seed}", end = "\t") if verbose else None

        params = {**best_params_noseed, "random_state": seed}

        model = xgb.XGBRegressor(**params)

        model.fit(X, y_log)
        pred = np.expm1(model.predict(X_test))
        preds.append(pred)

        rmse = calculate_cv(X, y_log, params)
        RMSE_cv.append(rmse)

        print(f"[DONE]\t{rmse:.5f}", end = "\n")  if verbose else None

    
    avg_cv_rmse = np.average(RMSE_cv)
    print(f"seed = {"END"}\t[MEAN]\t{avg_cv_rmse:.5f}")  if verbose else None

    return preds, avg_cv_rmse

In [138]:
multiple_preds, avg_rmse_cv = average_multiple_models(
    best_params, 
    # [seed_val for seed_val in range(5)],
    [42],
    X, y_log, X_test, 
    verbose=True
)

seed = 42	[DONE]	520.55161
seed = END	[MEAN]	520.55161


In [139]:
pred = np.mean(multiple_preds, axis=0)

len(pred)

365

# Generating Model Params

In [140]:
def print_model(model_type: str, params: dict[str, float], validation_rmse: float, columns_train_datset: list[str]) -> None:
    print(f"[{model_type}]", end="\n")
    print(f"V-RMSE: {validation_rmse:.2f}", end = "\n")

    print(f"params=[", end = "")
    for param_name, param_value in params.items():
        if type(param_value) is not str and type(param_value) is not int:
            print(f"{param_value:.4f}", end = ",")
        elif type(param_value) is not str:
            print(f"{param_value}", end = ",")
        else:
            print(f"'{param_value}'", end = ",")
    print(f"]", end = "\n")
    
    print(f"columns=[", end = "")
    for col_name in columns_train_datset:
        print(f"{col_name}", end = ",")
    print(f"]", end = "")

In [154]:
print_model(f"XGBoost + Optuna + Models Average (1) (seed=42) + round (1)", best_params, avg_rmse_cv, df_train_featured.columns.to_list())

[XGBoost + Optuna + Models Average (1) (seed=42) + round (1)]
V-RMSE: 520.55
params=[6,420,0.0300,0.6080,0.5460,0.7600,0.5870,5,0.0010,1.3000,0.1400,3.8780,'hist',42,]
columns=[temp,atemp,hum,wind,count,month,dow,quart,week,doy_norm,t,temp2,temp3,hum2,wind2,temp-atemp,tempxhum,tempxwind,atempxhum,humxwind,wcondxtemp,break_schools,break_academia,holiday,wcond_2,wcond_3,sind1,cosd1,sind2,cosd2,sind3,cosd3,sind4,cosd4,]

# Submission Generator

In [155]:
from scripts.save_submission import save_submission_csv

In [156]:
df_date = pd.read_csv(f"data-00-raw/bike_test.csv", parse_dates=["date"])

df_date["date"]

0     2023-01-01
1     2023-01-02
2     2023-01-03
3     2023-01-04
4     2023-01-05
         ...    
360   2023-12-27
361   2023-12-28
362   2023-12-29
363   2023-12-30
364   2023-12-31
Name: date, Length: 365, dtype: datetime64[us]

In [157]:
df_submission = pd.DataFrame({"date": df_date["date"], "pred": pred})

df_submission["pred"] = df_submission["pred"].round(1)

df_submission

,date,pred
0,2023-01-01,1282.500000
1,2023-01-02,1567.300049
2,2023-01-03,1309.400024
3,2023-01-04,1377.199951
4,2023-01-05,1798.300049
...,...,...
360,2023-12-27,867.000000
361,2023-12-28,1448.800049
362,2023-12-29,1921.900024
363,2023-12-30,1354.099976


In [158]:
save_submission_csv(df_submission)